In [ ]:
import pandas as pd
import numpy as np

#1. Upload the Spotify file

In [ ]:
from google.colab import files
uploaded = files.upload() #upload the file directly

# 2. Let's visualize the Spotify as DataFrame

In [ ]:
df = pd.read_csv('/content/spotify-data.csv')
df.head()

,id,name,artists,duration_ms,release_date,year,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence,mode,key,popularity,explicit
0,6KbQ3uYMLKb5jDxLF7wYDD,Singende Bataillone 1. Teil,['Carl Woitschach'],158648,1928,1928,0.995,0.708,0.1950,0.563,0.1510,-12.428,0.0506,118.469,0.7790,1,10,0,0
1,6KuQTIu1KoTTkLXKrwlLPV,"Fantasiestücke, Op. 111: Più tosto lento","['Robert Schumann', 'Vladimir Horowitz']",282133,1928,1928,0.994,0.379,0.0135,0.901,0.0763,-28.454,0.0462,83.972,0.0767,1,8,0,0
2,6L63VW0PibdM1HDSBoqnoM,Chapter 1.18 - Zamek kaniowski,['Seweryn Goszczyński'],104300,1928,1928,0.604,0.749,0.2200,0.000,0.1190,-19.924,0.9290,107.177,0.8800,0,5,0,0
3,6M94FkXd15sOAOQYRnWPN8,Bebamos Juntos - Instrumental (Remasterizado),['Francisco Canaro'],180760,9/25/28,1928,0.995,0.781,0.1300,0.887,0.1110,-14.734,0.0926,108.003,0.7200,0,1,0,0
4,6N6tiFZ9vLTSOIxkj8qKrd,"Polonaise-Fantaisie in A-Flat Major, Op. 61","['Frédéric Chopin', 'Vladimir Horowitz']",687733,1928,1928,0.990,0.210,0.2040,0.908,0.0980,-16.829,0.0424,62.149,0.0693,1,11,1,0


In [ ]:
popular_dict = df['popularity'].value_counts().to_dict()

sort_dict = dict(sorted(popular_dict.items()))
sort_dict
MAX = max(popular_dict.items())
MIN = min(popular_dict.items())
print("MAX = "+str(MAX))
print("MIN = "+str(MIN))

MAX = (100, 1)
MIN = (0, 27357)


### **NOTE:** We know that there will be 0-100 popularity indexs.

In [ ]:
musicKey_dict = df['key'].value_counts().to_dict()

sort_dict = dict(sorted(musicKey_dict.items()))
sort_dict

{0: 21499,
 1: 12816,
 2: 18821,
 3: 7185,
 4: 12921,
 5: 16336,
 6: 8586,
 7: 20757,
 8: 10711,
 9: 17628,
 10: 12056,
 11: 10593}

In [ ]:
mode_dict = df['mode'].value_counts().to_dict()

sort_dict = dict(sorted(mode_dict.items()))
sort_dict

{0: 49519, 1: 120390}

### **NOTE:** With MODE data, we can use binary model to predict MODE!!

# 2.1Prepare PySpark

In [ ]:
!pip install pyspark
!pip install findspark

In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql import SQLContext

In [ ]:
spark = SparkSession \
    .builder \
    .appName("classification") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext


sqlContext = SQLContext(sc)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
sdf = spark.createDataFrame(df)


In [ ]:
sdf.columns

['id',
 'name',
 'artists',
 'duration_ms',
 'release_date',
 'year',
 'acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']

In [ ]:
sdf.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- release_date: string (nullable = true)
 |-- year: long (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- mode: long (nullable = true)
 |-- key: long (nullable = true)
 |-- popularity: long (nullable = true)
 |-- explicit: long (nullable = true)



##Add Label column##

In [ ]:
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="mode", outputCol="label")
df = indexer.fit(sdf).transform(sdf)

In [ ]:
sdf.groupBy('mode').count().orderBy('count').show()

+----+------+
|mode| count|
+----+------+
|   0| 49519|
|   1|120390|
+----+------+



In [ ]:
df.groupBy('label').count().orderBy('count').show()

+-----+------+
|label| count|
+-----+------+
|  1.0| 49519|
|  0.0|120390|
+-----+------+



# 2.2 Prepare the features

In [ ]:
featureColumns = df.columns[5:]
featureColumns.remove('mode')
featureColumns.remove('year')
featureColumns.remove('label')
featureColumns

['acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'key',
 'popularity',
 'explicit']

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=featureColumns, outputCol="features") # outputCol has a default name: features.
assembled = assembler.transform(df)

In [ ]:
(trainingData, testData) = assembled.randomSplit([0.8,0.2], seed = 13234 )

##Model Using Logistic Regression + Evaluation##

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier,LogisticRegression
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(trainingData)

In [ ]:
predictions = model.transform(testData)

In [ ]:
predictions.show()

+--------------------+--------------------+--------------------+-----------+------------+----+------------+------------+------+----------------+--------+--------+-----------+-------+-------+----+---+----------+--------+-----+--------------------+--------------------+--------------------+----------+
|                  id|                name|             artists|duration_ms|release_date|year|acousticness|danceability|energy|instrumentalness|liveness|loudness|speechiness|  tempo|valence|mode|key|popularity|explicit|label|            features|       rawPrediction|         probability|prediction|
+--------------------+--------------------+--------------------+-----------+------------+----+------------+------------+------+----------------+--------+--------+-----------+-------+-------+----+---+----------+--------+-----+--------------------+--------------------+--------------------+----------+
|004ddQGTS8w7sDEKu...|          The Angels|['Melissa Etherid...|     280000|      1/1/89|1989|      

In [ ]:
predictions.select("features","rawprediction","probability","prediction", "label").show(10)

+--------------------+--------------------+--------------------+----------+-----+
|            features|       rawprediction|         probability|prediction|label|
+--------------------+--------------------+--------------------+----------+-----+
|[0.117,0.475,0.71...|[0.99507286698167...|[0.73008874332880...|       0.0|  0.0|
|[0.556,0.897,0.24...|[0.57646255510140...|[0.64025303477310...|       0.0|  0.0|
|[0.0948,0.756,0.5...|[0.66117385058136...|[0.65952402838441...|       0.0|  0.0|
|[0.055,0.374,0.67...|[1.15148170060551...|[0.75978145154117...|       0.0|  0.0|
|[0.564,0.335,0.57...|[0.72707195810028...|[0.67416240535804...|       0.0|  0.0|
|[0.456,0.47,0.715...|[1.61354942599011...|[0.83390359453389...|       0.0|  0.0|
|[0.847,0.451,0.35...|[0.88501802228285...|[0.70786099833489...|       0.0|  0.0|
|[0.98,0.46,0.292,...|[0.28640052222227...|[0.57111469370358...|       0.0|  1.0|
|[0.426,0.678,0.40...|[1.33541870355786...|[0.79173553934988...|       0.0|  0.0|
|[0.989,0.367,0.

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")

In [ ]:
accuracy = evaluator.evaluate(predictions)
print("Accuracy = %g " % (accuracy))

Accuracy = 0.710141 


In [ ]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
recall = evaluator.evaluate(predictions)
print("Recall =", recall)

Recall = 0.7101411012350025


In [ ]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator.evaluate(predictions)
print("F1 score = ", f1_score)

F1 score =  0.6016204421827177


In [ ]:
prediction_save=predictions.select("prediction", "label")

In [ ]:
metrics = MulticlassMetrics(prediction_save.rdd.map(tuple))

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
metrics.confusionMatrix().toArray().transpose()

array([[23918.,  9724.],
       [  157.,   290.]])

In [ ]:
#Save the best model
predictions.select("prediction", "label").write.save(path="predictions",
                                                     format="com.databricks.spark.csv",
                                                     header='true')

##Model Using Decision Tree Classifier + Evaluation##

In [ ]:
from pyspark.ml import Pipeline
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=5,minInstancesPerNode=20, impurity="gini")
pipeline = Pipeline(stages=[dt])
model = pipeline.fit(trainingData)

In [ ]:
predictions = model.transform(testData)

In [ ]:
predictions.show()

+--------------------+--------------------+--------------------+-----------+------------+----+------------+------------+------+----------------+--------+--------+-----------+-------+-------+----+---+----------+--------+-----+--------------------+-----------------+--------------------+----------+
|                  id|                name|             artists|duration_ms|release_date|year|acousticness|danceability|energy|instrumentalness|liveness|loudness|speechiness|  tempo|valence|mode|key|popularity|explicit|label|            features|    rawPrediction|         probability|prediction|
+--------------------+--------------------+--------------------+-----------+------------+----+------------+------------+------+----------------+--------+--------+-----------+-------+-------+----+---+----------+--------+-----+--------------------+-----------------+--------------------+----------+
|004ddQGTS8w7sDEKu...|          The Angels|['Melissa Etherid...|     280000|      1/1/89|1989|       0.117|  

In [ ]:
predictions.select("features","rawprediction","probability","prediction", "label").show(10)

+--------------------+-----------------+--------------------+----------+-----+
|            features|    rawprediction|         probability|prediction|label|
+--------------------+-----------------+--------------------+----------+-----+
|[0.117,0.475,0.71...|[17596.0,10293.0]|[0.63092975725196...|       0.0|  0.0|
|[0.556,0.897,0.24...| [15090.0,6941.0]|[0.68494394262629...|       0.0|  0.0|
|[0.0948,0.756,0.5...| [15090.0,6941.0]|[0.68494394262629...|       0.0|  0.0|
|[0.055,0.374,0.67...|[37746.0,10471.0]|[0.78283592923657...|       0.0|  0.0|
|[0.564,0.335,0.57...|    [622.0,395.0]|[0.61160275319567...|       0.0|  0.0|
|[0.456,0.47,0.715...|[37746.0,10471.0]|[0.78283592923657...|       0.0|  0.0|
|[0.847,0.451,0.35...| [20307.0,4838.0]|[0.80759594352754...|       0.0|  0.0|
|[0.98,0.46,0.292,...| [15090.0,6941.0]|[0.68494394262629...|       0.0|  1.0|
|[0.426,0.678,0.40...|[37746.0,10471.0]|[0.78283592923657...|       0.0|  0.0|
|[0.989,0.367,0.27...|[17596.0,10293.0]|[0.630929757

In [ ]:
prediction_save=predictions.select("features","rawprediction","probability","prediction", "label").show()

+--------------------+-----------------+--------------------+----------+-----+
|            features|    rawprediction|         probability|prediction|label|
+--------------------+-----------------+--------------------+----------+-----+
|[0.117,0.475,0.71...|[17596.0,10293.0]|[0.63092975725196...|       0.0|  0.0|
|[0.556,0.897,0.24...| [15090.0,6941.0]|[0.68494394262629...|       0.0|  0.0|
|[0.0948,0.756,0.5...| [15090.0,6941.0]|[0.68494394262629...|       0.0|  0.0|
|[0.055,0.374,0.67...|[37746.0,10471.0]|[0.78283592923657...|       0.0|  0.0|
|[0.564,0.335,0.57...|    [622.0,395.0]|[0.61160275319567...|       0.0|  0.0|
|[0.456,0.47,0.715...|[37746.0,10471.0]|[0.78283592923657...|       0.0|  0.0|
|[0.847,0.451,0.35...| [20307.0,4838.0]|[0.80759594352754...|       0.0|  0.0|
|[0.98,0.46,0.292,...| [15090.0,6941.0]|[0.68494394262629...|       0.0|  1.0|
|[0.426,0.678,0.40...|[37746.0,10471.0]|[0.78283592923657...|       0.0|  0.0|
|[0.989,0.367,0.27...|[17596.0,10293.0]|[0.630929757

In [ ]:
prediction_save=predictions.select("prediction", "label")

In [ ]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")


In [ ]:
accuracy = evaluator.evaluate(predictions)
print("Accuracy = %g " % (accuracy))

Accuracy = 0.722139 


In [ ]:
# Evaluate model performance
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Accuracy =", accuracy)

Accuracy = 0.7221391064566283


In [ ]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
recall = evaluator.evaluate(predictions)
print("Recall =", recall)


Recall = 0.7221391064566283


In [ ]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator.evaluate(predictions)
print("F1 score = ", f1_score)

F1 score =  0.6504478641184226


In [ ]:
metrics = MulticlassMetrics(prediction_save.rdd.map(tuple))

In [ ]:
metrics.confusionMatrix().toArray().transpose()

array([[23312.,  8709.],
       [  763.,  1305.]])

In [ ]:
print("summery")
print("====================")
print("Accuracy =", accuracy)
print("Recall =", recall)
print("F1 score = ", f1_score)

summery
Accuracy = 0.7221391064566283
Recall = 0.7221391064566283
F1 score =  0.6504478641184226


In [ ]:
sc.stop()